# Qwen3-14B + 自作LoRA を vLLM バックエンドで推論するノートブック

このノートブックは  を利用し、Qwen3-14B と自作 LoRA アダプタで推論する最小構成です。各セルは環境構築・パス設定・モデルロード・推論・クリーンアップに分かれています。


In [ ]:
# === 1. 環境構築: リポジトリ取得と依存インストール ===
import os
import subprocess
import sys

REPO_URL = 'https://github.com/fouga1221/llm-lab2.git'
DEFAULT_REPO_DIR = '/content/llm-lab2' if 'google.colab' in sys.modules else os.path.abspath('..')
REPO_DIR = os.environ.get('LLMLAB_REPO_DIR', DEFAULT_REPO_DIR)

if not os.path.exists(REPO_DIR):
    print('リポジトリをクローンします:', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('既存リポジトリを再利用します:', REPO_DIR)

requirements_path = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.exists(requirements_path):
    print('依存パッケージをインストールします (pip)。')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', requirements_path], check=False)
else:
    print('requirements.txt が見つかりませんでした。必要なパッケージを手動でインストールしてください。')


In [ ]:
# === 2. パス・モデル設定 ===
from pathlib import Path

REPO_ROOT = Path(REPO_DIR).resolve()
if 'google.colab' in sys.modules:
    DATA_ROOT = Path('/content/drive/MyDrive/llm-lab-runtime')
else:
    DATA_ROOT = Path.cwd() / 'runtime'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.environ.get('QWEN3_MODEL_NAME', 'Qwen/Qwen3-14B')
# 自作 LoRA のパス (例: Google Drive 上に配置)。必要に応じて環境変数で上書きしてください。
LORA_DIR = Path(os.environ.get('QWEN3_LORA_PATH', DATA_ROOT / 'loras' / 'qwen3_14b_custom'))
LORA_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'LORA_DIR: {LORA_DIR}')
print('LoRA 重みを以下のディレクトリに配置してください。存在しない場合はスキップされます。')


In [ ]:
# === 3. Python パス調整とバックエンド API の読み込み ===
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.llmlab.backends.vllm_backend import (
    ModelBundle,
    chat_loop,
    free_model,
    load_model,
    profile_generation,
)

print('バックエンド API をロードしました。')


In [ ]:
# === 4. vLLM モデルロード ===
cfg = {
    'model_name': MODEL_NAME,
    'tensor_parallel_size': int(os.environ.get('QWEN3_TP', 1)),
    'dtype': os.environ.get('QWEN3_DTYPE', 'auto'),
    'max_model_len': int(os.environ.get('QWEN3_MAX_LEN', 4096)),
    'gpu_memory_utilization': float(os.environ.get('QWEN3_GPU_UTIL', 0.90)),
    'download_dir': str(DATA_ROOT / 'model_cache'),
    'quantization': os.environ.get('QWEN3_QUANT', 'none'),
    'lora_path': str(LORA_DIR) if any(LORA_DIR.glob('**/*')) else None,
    'merge_lora': bool(int(os.environ.get('QWEN3_MERGE_LORA', '0'))),
}

print('ロード設定:', cfg)
bundle: ModelBundle = load_model(cfg)
print('モデルロード完了。利用可能なトークナイザ:', bool(bundle['tok']))
print('ロード時間 (秒):', bundle['cfg'].get('_timings', {}))


In [ ]:
# === 5. バッチ推論 & メトリクス計測 ===
import pandas as pd

prompts = [
    '以下の仕様を要約してください:
- 多段 LoRA でチューニングした Qwen3-14B
- 連携するエッジ端末向けアプリへの適用',
    'LoRA 適用済みモデルとして、顧客向け FAQ 生成ボットの導入メリットを3点で説明してください。',
]

outputs, metrics = profile_generation(
    bundle,
    prompts,
    max_new_tokens=int(os.environ.get('QWEN3_MAX_NEW', 256)),
    temperature=float(os.environ.get('QWEN3_TEMP', 0.7)),
    top_p=float(os.environ.get('QWEN3_TOP_P', 0.9)),
    repetition_penalty=float(os.environ.get('QWEN3_REP_PEN', 1.05)),
    stop=['
User:'],
)

display(pd.DataFrame({'prompt': prompts, 'output': outputs}))
display(pd.DataFrame([metrics]).T.rename(columns={0: 'value'}))


In [ ]:
# === 6. 対話モード (任意) ===
print('対話を開始します。終了コマンド: /exit')
try:
    chat_loop(
        bundle,
        system_prompt='あなたは Qwen3-14B ベースのアシスタントです。',
        stop_phrases=['/exit', ':q'],
        max_new_tokens=256,
        temperature=0.7,
    )
finally:
    print('対話モードを終了しました。')


In [ ]:
# === 7. 後片付け ===
free_model(bundle)
print('メモリを解放しました。')
